# Multi-Tenant Coexistence Demo

Demonstrates that multiple tenants coexist in the same Neptune cluster with complete
data isolation — no cross-contamination between tenants.

## How Multi-Tenancy Works

document-graph (and lexical-graph) use **label-based tenant scoping**:

```
Default tenant:  `User`           (no suffix)
Tenant "alpha":  `Useralpha__`    (label + tenant_id + __)
Tenant "beta":   `Userbeta__`     (label + tenant_id + __)
```

Both packages use the same format (via `TenantId.format_label()`), so:
- document-graph nodes for tenant "alpha" are isolated from tenant "beta"
- lexical-graph entities for tenant "alpha" are isolated from tenant "beta"
- Both packages' nodes for the SAME tenant can be queried together

## What This Notebook Proves

1. Two tenants writing identical node IDs don't collide
2. Queries are scoped — each tenant sees only its own data
3. An admin view can see all tenants when needed

**Prerequisites:**
- `pip install graphrag-toolkit-document-graph[graphrag]`
- Neptune cluster endpoint (see 00-Setup)


In [ ]:
# Setup
import json
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.document_graph.graph_build import node_to_cypher
from graphrag_toolkit.document_graph.query import DocumentGraphQueryEngine
from graphrag_toolkit.document_graph import Node

GRAPH_STORE = 'neptune-db://<your-neptune-cluster-endpoint>:8182'

gs = GraphStoreFactory.for_graph_store(GRAPH_STORE).__enter__()
print('Neptune connected')


## Step 1: Write Data for Tenant A

Create nodes under tenant `alpha`. These nodes get labels like `Useralpha__`
and are invisible to other tenants.


In [ ]:
TENANT_A = 'acme_corp'

tenant_a_data = [
    Node(id='u1', labels=['User'], properties={'name': 'Alice', 'role': 'admin', 'department': 'Engineering'}),
    Node(id='u2', labels=['User'], properties={'name': 'Bob', 'role': 'developer', 'department': 'Engineering'}),
    Node(id='p1', labels=['Project'], properties={'name': 'Apollo', 'status': 'active'}),
]

for node in tenant_a_data:
    cypher, params = node_to_cypher(node, tenant_id=TENANT_A)
    gs.execute_query(cypher, params)

print(f'Tenant A ({TENANT_A}): wrote {len(tenant_a_data)} nodes')


## Step 2: Write Data for Tenant B

Create nodes with the **same IDs** under tenant `beta`. Labels become `Userbeta__`.
Despite identical node IDs, there's no collision — different labels = different nodes.


In [ ]:
TENANT_B = 'globex_inc'

tenant_b_data = [
    Node(id='u1', labels=['User'], properties={'name': 'Charlie', 'role': 'manager', 'department': 'Sales'}),
    Node(id='u2', labels=['User'], properties={'name': 'Diana', 'role': 'analyst', 'department': 'Finance'}),
    Node(id='p1', labels=['Project'], properties={'name': 'Zeus', 'status': 'planning'}),
]

for node in tenant_b_data:
    cypher, params = node_to_cypher(node, tenant_id=TENANT_B)
    gs.execute_query(cypher, params)

print(f'Tenant B ({TENANT_B}): wrote {len(tenant_b_data)} nodes')


## Step 3: Query — Tenant Isolation

Query each tenant separately using `DocumentGraphQueryEngine` with a `tenant_id`.
Each tenant sees only its own data. No leakage.


In [ ]:
# Query Tenant A only
engine_a = DocumentGraphQueryEngine(gs, tenant_id=TENANT_A)
users_a = engine_a.get_nodes('User')
print(f'Tenant A Users: {len(users_a)}')
for u in users_a:
    props = u['n']['~properties']
    print(f'  {props.get("name")} ({props.get("role")})')

# Query Tenant B only
engine_b = DocumentGraphQueryEngine(gs, tenant_id=TENANT_B)
users_b = engine_b.get_nodes('User')
print(f'\nTenant B Users: {len(users_b)}')
for u in users_b:
    props = u['n']['~properties']
    print(f'  {props.get("name")} ({props.get("role")})')


## Step 4: Prove Isolation — Same ID, Different Tenants

Demonstrate that node `user-001` in tenant "alpha" has different properties
than node `user-001` in tenant "beta" — they're completely separate entities
that happen to share an ID.


In [ ]:
# Both tenants have node id='u1' but they're different nodes
a_u1 = engine_a.find_by_property('User', 'id', 'u1')
b_u1 = engine_b.find_by_property('User', 'id', 'u1')

a_name = a_u1[0]['n']['~properties']['name'] if a_u1 else 'NOT FOUND'
b_name = b_u1[0]['n']['~properties']['name'] if b_u1 else 'NOT FOUND'

print(f'Node u1 in Tenant A: {a_name}')
print(f'Node u1 in Tenant B: {b_name}')
print(f'\nSame ID, different data = tenant isolation working')


## Step 5: Cross-Tenant View (Admin)

An admin can query across all tenants by not specifying a `tenant_id` —
or by using a wildcard label query. This is for management/debugging only.


In [ ]:
# Admin can see all tenants by querying without tenant filter
all_users = gs.execute_query('MATCH (n) WHERE labels(n)[0] CONTAINS "User" RETURN labels(n) as lbl, properties(n) as props LIMIT 10')
print(f'All User nodes across tenants: {len(all_users)}')
for u in all_users:
    label = u['lbl'][0]
    name = u['props'].get('name', '?')
    # Extract tenant from label: Usertenant__
    parts = label.split('__')
    tenant = parts[2] if len(parts) >= 3 else '?'
    print(f'  {name} (tenant: {tenant})')


# Clean up
gs.__exit__(None, None, None)


## Summary

| Aspect | How it works |
|--------|-------------|
| **Isolation** | Labels encode tenant: `Useracme_corp__` vs `Userglobex_inc__` |
| **Same IDs** | Node id='u1' exists in both tenants independently |
| **Querying** | `DocumentGraphQueryEngine(gs, tenant_id=X)` scopes all queries |
| **Admin view** | Query without tenant filter sees all data |
| **No leakage** | Tenant A cannot see Tenant B's data through normal queries |
